# Examples for paper **"Computing Burchnall-Chaundy Ideal in elleiptic extensions"**

In [1]:
import sys
sys.path.insert(0, "..") # dalgebra is here

from dalgebra import *
from dalgebra.commutators import *
from dalgebra.commutators.commutator import BC_ideal

import logging
logging.getLogger("dalgebra").setLevel(logging.INFO)

def eval_op_polynomial(poly, z, **kwds):
    return sum(coeff*eval_op_mon(mon,z,**kwds) for (coeff, mon) in zip(poly.coefficients(), poly.monomials()))

def eval_op_mon(mon, z, **kwds):
    variables = [kwds[str(v)].sym_power(mon.degree(v), z) for v in mon.variables()]
    return reduce(lambda p, q : p.dot(q, z), variables) if len(variables) > 0 else 1

def reduce_by_curve(operator, curve, gen):
    coefficients = [operator.coefficient_full(gen[i]) for i in range(operator.order(gen)+1)]
    zero_conditions = [coeff.conditions_to_zero() for coeff in coefficients]
    reduced_zero = [[(mon, curve.reduce(coeff.numerator())/curve.reduce(coeff.denominator())) for (mon,coeff) in condition] for condition in zero_conditions]
    reduced_coeff = [sum(mon*coeff for (mon,coeff) in condition) for condition in reduced_zero]
    return sum(gen[i]*coeff for i,coeff in enumerate(reduced_coeff))

## . Ideal for operator with $n=4$ and elliptic coefficients

Let us consider an operator of order 4 with coefficients in the differential field $\mathbb{Q}\langle \eta\rangle$ where 
$$\eta'^2 = \eta^3 + 1,$$
which can be seen as a specific case of the generic elliptic extension where $\eta'^2 = \eta^3 + g_2 \eta + g_3$.

We consider the following differential operator:
$$\mathbb{L} = -\left(\frac{9}{2} \eta^{2}\right) - \left(6 \eta'\right)\partial - \left(3 \eta\right)\partial^2 + \partial^4.$$

We start by computing the centralizer of this operator, so we can see it is generated (as a $\mathbb{Q}[\mathbb{L}]$-module) by operators of order $0$, $5$, $6$ and $7$.

### Computing the centralizer as module

In [2]:
B = DifferentialRing(QQ)
D.<eta> = DElliptic(B, "eta_p^2 - eta^3 - 1")
eta_p = eta.derivative()
Us = (-9/2*eta^2, -6*eta_p, -3*eta)
L, (_,G_1,G_2,G_3),_ = GetCentralizer(Us, 10, starting_level=5, ignore_bound=True)
z = L.parent().gen("z")

2025-07-23 08:38:23 INFO     No operation is given: we set a zero derivative.


In [3]:
G_1

-(45/8*eta*eta_p)*z_0 - (45/4*eta^2)*z_1 - (75/8*eta_p)*z_2 - (15/4*eta)*z_3 + z_5

In [4]:
G_2

-(45/4*eta^3 + 9/2)*z_0 - (45/2*eta*eta_p)*z_1 - (45/2*eta^2)*z_2 - (27/2*eta_p)*z_3 - (9/2*eta)*z_4 + z_6

In [5]:
G_3

-(945/32*eta^2*eta_p)*z_0 - (945/16*eta^3 + 405/16)*z_1 - (945/16*eta*eta_p)*z_2 - (315/8*eta^2)*z_3 - (147/8*eta_p)*z_4 - (21/4*eta)*z_5 + z_7

### Computing the BC-ideal

Following the algorithm in the paper, let us consider the polynomial ring $\mathbb{Q}[\lambda,\mu_1,\mu_2,\mu_3]$ and fix the monomial order defined by blocks where
* $\lambda < \{\mu_1,\mu_2,\mu_3\}$,
* In the bigger block, we use a weighted lexicografic order where we use the following weighted total degree:
  $$w(p(\mu_1,\mu_2,\mu_3)) = 5\deg_{\mu_1}(p) + 6\deg_{\mu_2}(p) + 7\deg_{\mu_3}(p).$$

Then we compute the representation of each biproduct $G_iG_j$ as elements in the $\mathbb{Q}[L]$-module:

In [6]:
G_1.dot(G_1, z) - L.dot(G_2, z) + 27/64*L # \mu_1^2 - \lambda\mu_2 + 27/64\lambda

0

In [7]:
G_1.dot(G_2, z) - L.dot(G_3, z) # \mu_1\mu_2 - \lambda\mu_3

0

In [8]:
G_1.dot(G_3, z) - L.sym_power(3, z) + 27/64*G_2 # \mu_1\mu_3 - \lambda^3 + 27/64\mu_2

0

In [9]:
G_2.dot(G_2, z) - L.sym_power(3, z) # \mu_2^2 - \lambda^3

0

In [10]:
G_2.dot(G_3, z) - L.dot(L.dot(G_1, z), z) # \mu_2\mu_3 - \lambda^2\mu_1

0

In [11]:
G_3.dot(G_3, z) - L.dot(L.dot(G_2, z), z) + 27/64*L.dot(L, z) # \mu_3^2 - \lambda^2\mu_2 + 27/64\lambda^2

0

Hence, the Burchnall-Chaundy ideal for the operator $\mathbb{L}$ described above (with elliptic coefficients) is generated by the polynomials:
$$BC(\mathbb{L}) = \left(\mu_1^2 - \lambda\mu_2 + 27/64\lambda, \mu_1\mu_2 - \lambda\mu_3, \mu_1\mu_3 - \lambda^3 + 27/64\mu_2, \mu_2^2 - \lambda^3, \mu_2\mu_3 - \lambda^2\mu_1, \mu_3^2 - \lambda^2\mu_2 + 27/64\lambda^2\right),$$
which are six polynomials that provide plenty of algebraic relations between the variables.

In [12]:
BC_R.<lambda_, mu_1, mu_2, mu_3> = QQ[]
l, m_1, m_2, m_3 = BC_R.gens()

I = ideal(ideal([
    mu_1^2 - lambda_*mu_2 + 27/64*lambda_,
    mu_1*mu_2 - lambda_*mu_3, 
    mu_1*mu_3 - lambda_^3 + 27/64*mu_2, 
    mu_2^2 - lambda_^3, 
    mu_2*mu_3 - lambda_^2*mu_1, 
    mu_3^2 - lambda_^2*mu_2 + 27/64*lambda_^2
]).groebner_basis())

In [13]:
show(I)

Ideal (lambda_^3 - mu_1*mu_3 - 27/64*mu_2, lambda_^2*mu_1 - mu_2*mu_3, lambda_^2*mu_2 - 27/64*lambda_^2 - mu_3^2, mu_1^2 - lambda_*mu_2 + 27/64*lambda_, mu_1*mu_2 - lambda_*mu_3, mu_2^2 - mu_1*mu_3 - 27/64*mu_2) of Multivariate Polynomial Ring in lambda_, mu_1, mu_2, mu_3 over Rational Field

This result can be achieved directly using the method `BC_ideal` in the module `dalgebra.commutators.commutator`:

In [14]:
I = BC_ideal(L, [G_1,G_2,G_3], L.parent().gen("z"))
show(I)
BC_R = I.ring()
show(BC_R)

2025-07-23 08:38:30 WARNING  Method constant_ring() not properly implemented for D-Extensions.
2025-07-23 08:38:30 WARNING  Method constant_ring() not properly implemented for D-Extensions.
2025-07-23 08:38:30 WARNING  Method constant_ring() not properly implemented for D-Extensions.
2025-07-23 08:38:30 WARNING  Method constant_ring() not properly implemented for D-Extensions.
2025-07-23 08:38:30 WARNING  Method constant_ring() not properly implemented for D-Extensions.
2025-07-23 08:38:30 WARNING  Method constant_ring() not properly implemented for D-Extensions.
2025-07-23 08:38:30 WARNING  Method constant_ring() not properly implemented for D-Extensions.


Ideal (mu_1^2 - mu_2*lambda_ + 27/64*lambda_, mu_1*mu_2 - mu_3*lambda_, mu_1*mu_3 + 27/64*mu_2 - lambda_^3, mu_2^2 - lambda_^3, mu_2*mu_3 - mu_1*lambda_^2, mu_3^2 - mu_2*lambda_^2 + 27/64*lambda_^2) of Multivariate Polynomial Ring in mu_1, mu_2, mu_3, lambda_ over Rational Field

Multivariate Polynomial Ring in mu_1, mu_2, mu_3, lambda_ over Rational Field

IOStream.flush timed out
IOStream.flush timed out


### Checking that spectral curves are in the ideal

In [16]:
from dalgebra.commutators.spectral import *
spectral_ops = spectral_operators(G_1,G_2,G_3,L, names=["mu_1", "mu_2", "mu_3","lambda_"])
spectral_curves = {(i,j) : spectral_ops[i].sylvester_resultant(spectral_ops[j], "z") for i in range(len(spectral_ops)) for j in range(i+1,len(spectral_ops))}
all(I.reduce(BC_R(value)) == 0 for value in spectral_curves.values())

True

Now we check the opposite (reducing the BC ideal by the resultants):

In [17]:
I_res = ideal(BC_R, list(spectral_curves.values()))
I_rad = ideal(I_res.radical().groebner_basis())

In [18]:
[I_rad.reduce(p) for p in I.basis]

[mu_1^2 - mu_2*lambda_ + 27/64*lambda_,
 mu_1*mu_2 - mu_3*lambda_,
 64/81*mu_1^2*lambda_^2 - 64/81*mu_2*lambda_^3 + 1/3*lambda_^3,
 0,
 mu_2*mu_3 - mu_1*lambda_^2,
 mu_1^2*lambda_ - mu_2*lambda_^2 + 27/64*lambda_^2]

### Computing the right factor and comparing

We can compute the common right factor looking to the subresultant sequences:

In [19]:
G1_, G2_, G3_, L_  = spectral_ops
SO_R = L_.parent()
z_ = SO_R.gen("z")
SO_R

Ring of operator polynomials in (z) over D-Elliptic extension of Fraction Field of Differential Ring [[Multivariate Polynomial Ring in mu_1, mu_2, mu_3, lambda_ over Rational Field], (0,)] with element eta whose derivative satisfies 
	[eta_p^2 - eta^3 - 1 = 0]

In [20]:
lg1_srs = [reduce_by_curve(op, I, z_) for op in SO_R.sylvester_subresultant_sequence(L_,G1_,z_)]
lg2_srs = [reduce_by_curve(op, I, z_) for op in SO_R.sylvester_subresultant_sequence(L_,G2_,z_)]
lg3_srs = [reduce_by_curve(op, I, z_) for op in SO_R.sylvester_subresultant_sequence(L_,G3_,z_)]

In [21]:
gcrf_L = lg1_srs[1], lg2_srs[2], lg3_srs[1]
gcrf_L

(((27/16*lambda_^2*eta + 9/8*mu_2*lambda_)*eta_p + 9/4*mu_1*lambda_*eta^2 - mu_1*lambda_^2)*z_0 - (3/2*mu_1*lambda_*eta_p + 9/8*lambda_^2*eta^2 + 3/4*mu_2*lambda_*eta - lambda_^3)*z_1,
 (27/8*lambda_*eta^3 + 9/4*mu_2*eta^2 - 3/2*lambda_^2*eta - mu_2*lambda_)*z_0 + ((9/4*lambda_*eta - 3/2*mu_2)*eta_p)*z_1 - (9/4*lambda_*eta^2 - lambda_^2)*z_2,
 ((27/16*mu_2*lambda_^2*eta + 9/8*lambda_^4)*eta_p + 9/4*mu_3*lambda_^2*eta^2 - mu_3*lambda_^3)*z_0 - (3/2*mu_3*lambda_^2*eta_p + 9/8*mu_2*lambda_^2*eta^2 + 3/4*lambda_^4*eta - mu_2*lambda_^3)*z_1)

Now we have the three common right factors of $L$ with the other generators of Goodearl basis. We have two operators of order 1 and one operator of order 2. How can we compare the solutions?

For the order 1 operators, it is enough to check whether the $2\times 2$ determinant with the coefficients (i.e., whether the operators are proportional or not.

For comparing the order 2 operator with the other two, we need comething else. However, resultants come to rescue: if the resultant is zero, then two operators share a solution. Hence, if we are comparing an order 1 operator with an order 2, then the resultant to be zero is equivalent to the smaller operator divides the bigger or, equivalently, the solution of the order 1 operator is a solution of the order 2 operator.

In [22]:
# Comparing the order 1 operators
det = gcrf_L[0].coefficient_full(z_[0])*gcrf_L[2].coefficient_full(z_[1]) - gcrf_L[2].coefficient_full(z_[0])*gcrf_L[0].coefficient_full(z_[1])
reduce_by_curve(det*z_[0], I, z_)

0

In [23]:
# Comparing the order 2 operator
res = SO_R.sylvester_resultant(gcrf_L[0], gcrf_L[1], z_)
reduce_by_curve(res*z_[0], I, z_)

0

Hence, we conclude there is a common solution to all differential operators in the centralizer of $L$.

### Analyzing the curve in Sage

In [24]:
A.<m_1,m_2,m_3,l> = AffineSpace(QQ,4)
local_dict = {'l':l, 'm_1':m_1, 'm_2':m_2, 'm_3':m_3}
els = [sage_eval(str(el).replace("lambda_", "l").replace("mu_1", "m_1").replace("mu_2", "m_2").replace("mu_3","m_3"), locals=local_dict) for el in I.gens()]
curve = Curve(els, A)
curve.genus()

1

## . Ideal for operator with $n=4$ and elliptic coefficients (second example)

Let us consider an operator of order 4 with coefficients in the differential field $\mathbb{Q}\langle \eta\rangle$ where 
$$\eta'^2 = \eta^3 + 1,$$
which can be seen as a specific case of the generic elliptic extension where $\eta'^2 = \eta^3 + g_2 \eta + g_3$.

We consider the following differential operator:
$$\mathbb{L} = -\left(15 \eta'\right)\partial - \left(9 \eta\right) + \partial^4.$$

We start by computing the centralizer of this operator, so we can see it is generated (as a $\mathbb{Q}[\mathbb{L}]$-module) by operators of order $0$, $5$, $10$ and $11$.

### Computing the centralizer as module

In [25]:
B = DifferentialRing(QQ)
D.<eta> = DElliptic(B, "eta_p^2 - eta^3 - 1")
eta_p = eta.derivative()
Us = (0, -15*eta_p, -9*eta)
L, (_,G_1,G_2,G_3),_ = GetCentralizer(Us, 10, starting_level=5, ignore_bound=True)
z = L.parent().gen("z")

2025-06-25 11:24:20 INFO     No operation is given: we set a zero derivative.


In [26]:
G_1

(315/16*eta*eta_p)*z_0 - (45/8*eta^2)*z_1 - (195/8*eta_p)*z_2 - (45/4*eta)*z_3 + z_5

In [27]:
print(G_2)
G_2_ = G_2[1]

([0, 2, 0, 0], ((51975/16*eta^3 + 39825/64)*eta_p)*z_1 + (4725*eta^4 + 186975/64*eta)*z_2 + (23625/8*eta^2*eta_p)*z_3 + (1575/2*eta^3 + 14625/64)*z_4 - (315/2*eta*eta_p)*z_5 - (945/4*eta^2)*z_6 - (105*eta_p)*z_7 - (45/2*eta)*z_8 + z_10)


In [28]:
G_3

(56133/2048*eta*eta_p)*z_0 + (12474*eta^5 + 9472221/1024*eta^2)*z_1 + ((1232847/64*eta^3 + 3396195/1024)*eta_p)*z_2 + (434511/32*eta^4 + 4221261/512*eta)*z_3 + (168399/32*eta^2*eta_p)*z_4 + (11781/16*eta^3 + 13167/128)*z_5 - (3465/8*eta*eta_p)*z_6 - (693/2*eta^2)*z_7 - (1023/8*eta_p)*z_8 - (99/4*eta)*z_9 + z_11

### Computing the BC-ideal

Following the algorithm in the paper, let us consider the polynomial ring $\mathbb{Q}[\lambda,\mu_1,\mu_2,\mu_3]$ and fix the monomial order defined by blocks where
* $\lambda < \{\mu_1,\mu_2,\mu_3\}$,
* In the bigger block, we use a weighted lexicografic order where we use the following weighted total degree:
  $$w(p(\mu_1,\mu_2,\mu_3)) = 5\deg_{\mu_1}(p) + 6\deg_{\mu_2}(p) + 7\deg_{\mu_3}(p).$$

Then we compute the representation of each biproduct $G_iG_j$ as elements in the $\mathbb{Q}[L]$-module:

In [29]:
G_1.dot(G_1, z) - G_2_ # \mu_1^2 - \mu_2 --> this was known already

0

In [30]:
G_1.dot(G_2_, z) - G_3.dot(L, z) + 1053/128*G_1.dot(L, z) # \mu_1\mu_2 - \mu_3\lambda + (1053/128)\mu_1\lambda = \mu_1^3 - \mu_3\lambda + (1053/128)\mu_1\lambda

0

In [31]:
G_1.dot(G_3, z) - L.sym_power(4, z) + 351/128*G_1.dot(G_1, z) - 250047/4096*L # \mu_1\mu_3 - \lambda^4 + (351/128)\mu_1^2 - (250047/4096)\lambda

0

In [32]:
G_2_.dot(G_2_, z) - L.sym_power(5, z) + 351/32*G_2_.dot(L, z) - 250047/4096*L.dot(L,z) # \mu_2^2 - \lambda^5 + (351/32)\mu_2\lambda - 250047/4096\lambda^2

0

In [33]:
G_2_.dot(G_3, z) - G_1.dot(L.sym_power(4, z), z) + 351/128*G_3.dot(L,z) - 1369791/16384*G_1.dot(L,z) # \mu_2\mu_3 - \mu_1\lambda^4 + (351/128)\mu_3\lambda - (1396791/16384)\mu_1\lambda

0

In [34]:
G_3.dot(G_3, z) - G_2_.dot(L.sym_power(3,z),z) - 351/64*L.sym_power(4,z) - 1123389/16384*G_2_ - 87766497/262144*L # \mu_3^2 - \mu_2\lambda^3 - (351/64)\lambda^4 - 1123389/16384\mu_2 - 87766497/262144\lambda

0

Hence, the Burchnall-Chaundy ideal for the operator $\mathbb{L}$ described above (with elliptic coefficients) is generated by the polynomials:
$$\begin{array}{rcl}
    BC(\mathbb{L}) &{}={}& \left(\mu_1^2 - \mu_2,\right.\\
                   & & \mu_1\mu_2 - \mu_3\lambda + (1053/128)\mu_1\lambda,\\
                   & & \mu_1\mu_3 - \lambda^4 + (351/128)\mu_1^2 - (250047/4096)\lambda,\\
                   & & \mu_2^2 - \lambda^5 + (351/32)\mu_2\lambda - (250047/4096)\lambda^2,\\
                   & & \mu_2\mu_3 - \mu_1\lambda^4 + (351/128)\mu_3\lambda - (1369791/16384)\mu_1\lambda,\\
                   & &\left.\mu_3^2 - \mu_2\lambda^3 - (351/64)\lambda^4 - (1123389/16384)\mu_2 - (87766497/262144)\lambda\right).
\end{array}$$
which are six polynomials that provide plenty of algebraic relations between the variables.

In [35]:
BC_R = PolynomialRing(QQ, ("mu_1","mu_2","mu_3","lambda_"), order=TermOrder("wdegrevlex", (5,10,11)) + TermOrder("deglex", 1))
mu_1, mu_2, mu_3,lambda_ = BC_R.gens()

I = ideal(ideal([
    mu_1^2 - mu_2,
    mu_1*mu_2 - mu_3*lambda_ + (1053/128)*mu_1*lambda_, 
    mu_1*mu_3 - lambda_^4 + (351/128)*mu_2 - (250047/4096)*lambda_, 
    mu_2^2 - lambda_^5 + (351/32)*mu_2*lambda_ - 250047/4096*lambda_^2, 
    mu_2*mu_3 - mu_1*lambda_^4 + (351/128)*mu_3*lambda_ - (1369791/16384)*mu_1*lambda_, 
    mu_3^2 - mu_2*lambda_^3 - (351/64)*lambda_^4 - 1123389/16384*mu_2 - 87766497/262144*lambda_
]).groebner_basis())

In [36]:
all(eval_op_polynomial(poly, z, lambda_=L, mu_1=G_1, mu_2=G_2_, mu_3=G_3) == 0 for poly in I.gens())

True

This result can be achieved directly using the method `BC_ideal` in the module `dalgebra.commutators.commutator`:

In [37]:
I = BC_ideal(L, [G_1,G_2,G_3], L.parent().gen("z"))
show(I)
BC_R = I.ring()
show(BC_R)

2025-06-25 11:25:26 WARNING  Method constant_ring() not properly implemented for D-Extensions.
2025-06-25 11:25:26 WARNING  Method constant_ring() not properly implemented for D-Extensions.
2025-06-25 11:25:26 WARNING  Method constant_ring() not properly implemented for D-Extensions.
2025-06-25 11:25:26 WARNING  Method constant_ring() not properly implemented for D-Extensions.
2025-06-25 11:25:26 WARNING  Method constant_ring() not properly implemented for D-Extensions.
2025-06-25 11:25:26 WARNING  Method constant_ring() not properly implemented for D-Extensions.
2025-06-25 11:25:27 WARNING  Method constant_ring() not properly implemented for D-Extensions.


Ideal (mu_1^2 - mu_2, mu_1*mu_2 - mu_3*lambda_ + 1053/128*mu_1*lambda_, mu_1*mu_3 + 351/128*mu_2 - lambda_^4 - 250047/4096*lambda_, mu_2^2 + 351/32*mu_2*lambda_ - lambda_^5 - 250047/4096*lambda_^2, mu_2*mu_3 + 351/128*mu_3*lambda_ - mu_1*lambda_^4 - 1369791/16384*mu_1*lambda_, mu_3^2 - mu_2*lambda_^3 - 1123389/16384*mu_2 - 351/64*lambda_^4 - 87766497/262144*lambda_) of Multivariate Polynomial Ring in mu_1, mu_2, mu_3, lambda_ over Rational Field

Multivariate Polynomial Ring in mu_1, mu_2, mu_3, lambda_ over Rational Field

In [39]:
show(ideal(I.groebner_basis()))

Ideal (mu_3^2 - mu_2*lambda_^3 - 1123389/16384*mu_2 - 351/64*lambda_^4 - 87766497/262144*lambda_, mu_2*mu_3 + 351/128*mu_3*lambda_ - mu_1*lambda_^4 - 1369791/16384*mu_1*lambda_, mu_2^2 + 351/32*mu_2*lambda_ - lambda_^5 - 250047/4096*lambda_^2, mu_1*mu_3 + 351/128*mu_2 - lambda_^4 - 250047/4096*lambda_, mu_1*mu_2 - mu_3*lambda_ + 1053/128*mu_1*lambda_, mu_1^2 - mu_2) of Multivariate Polynomial Ring in mu_1, mu_2, mu_3, lambda_ over Rational Field

### Checking that spectral curves are in the ideal

In [38]:
from dalgebra.commutators.spectral import *
spectral_ops = spectral_operators(G_1,G_2_,G_3,L,  names=["mu_1", "mu_2", "mu_3","lambda_"])
spectral_curves = {(i,j) : spectral_ops[i].sylvester_resultant(spectral_ops[j], "z") for i in range(len(spectral_ops)) for j in range(i+1,len(spectral_ops))}
all(I.reduce(BC_R(value))==0 for value in spectral_curves.values())

True

Now we check the opposite (reducing the BC ideal by the resultants):

In [ ]:
I_res = ideal(BC_R, list(spectral_curves.values()))
I_rad = ideal(I_res.radical().groebner_basis())

In [ ]:
[I_rad.reduce(p) for p in I.basis]

### Computing the right factor and comparing

We can compute the common right factor looking to the subresultant sequences:

In [40]:
G1_, G2_, G3_, L_  = spectral_ops
SO_R = L_.parent()
z_ = SO_R.gen("z")
SO_R

Ring of operator polynomials in (z) over D-Elliptic extension of Fraction Field of Differential Ring [[Multivariate Polynomial Ring in mu_1, mu_2, mu_3, lambda_ over Rational Field], (0,)] with element eta whose derivative satisfies 
	[eta_p^2 - eta^3 - 1 = 0]

In [41]:
L_.order(z_)

4

In [42]:
G1_.order(z_), G2_.order(z_), G3_.order(z_)

(5, 10, 11)

In [43]:
lg1_srs = [reduce_by_curve(op, I, z_) for op in SO_R.sylvester_subresultant_sequence(L_,G1_,z_)]; print("Done (0,1)", end="\r")
lg2_srs = [reduce_by_curve(op, I, z_) for op in SO_R.sylvester_subresultant_sequence(L_,G2_,z_)]; print("Done (0,2)", end="\r")
lg3_srs = [reduce_by_curve(op, I, z_) for op in SO_R.sylvester_subresultant_sequence(L_,G3_,z_)]; print("Done (0,3)", end="\r")

In [44]:
gcrf_L = lg1_srs[1], lg2_srs[2], lg3_srs[1]
show(gcrf_L)

(((6615/64*lambda_*eta^3 + 63/8*lambda_^2*eta + 21/8*mu_2 + 1323/512*lambda_)*eta_p + 945/16*mu_1*eta^4 + 27/4*mu_1*lambda_*eta^2 + 189/8*mu_1*eta - mu_1*lambda_^2)*z_0 + ((-189/8*mu_1*eta^2 - 3*mu_1*lambda_)*eta_p - 1323/32*lambda_*eta^4 + (-9/4*mu_2 - 6615/256*lambda_)*eta + lambda_^3)*z_1,
 (3969/32*lambda_^2*eta^5 + (945/16*mu_2 + 413343/1024*lambda_)*eta^4 + 441/8*lambda_^3*eta^3 + (27/4*mu_2*lambda_ + 35397/256*lambda_^2)*eta^2 + (-9/2*lambda_^4 + 189/8*mu_2 - 260253/512*lambda_)*eta - mu_2*lambda_^2 - 63/64*lambda_^3)*z_0 + ((11907/64*lambda_*eta^5 + 189/2*lambda_^2*eta^3 + (-189/8*mu_2 - 11907/512*lambda_)*eta^2 + 9*lambda_^3*eta - 3*mu_2*lambda_ - 189/64*lambda_^2)*eta_p)*z_1 - (3969/64*lambda_*eta^6 + 189/4*lambda_^2*eta^4 + 567/8*lambda_*eta^3 + 9*lambda_^3*eta^2 + 27*lambda_^2*eta - lambda_^4 - 567/8*lambda_)*z_2,
 (((6615/64*mu_2*lambda_^3 + 6965595/8192*lambda_^4 + 7431218235/1048576*mu_2 + 7825072801455/134217728*lambda_)*eta^3 + (63/8*mu_2*lambda_^4 + 66339/1024*lambda_^5 + 70773507/131072*mu_2*lambda_ + 74524502871/16777216*lambda_^2)*eta + 21/8*lambda_^7 - 4725/1024*mu_2*lambda_^3 + 47381355/131072*lambda_^4 - 5308013025/16777216*mu_2 + 13362816630177/1073741824*lambda_)*eta_p + (945/16*mu_3*lambda_^3 + 1061602605/262144*mu_3)*eta^4 + (27/4*mu_3*lambda_^4 + 30331503/65536*mu_3*lambda_)*eta^2 + (189/8*mu_3*lambda_^3 + 212320521/131072*mu_3)*eta - mu_3*lambda_^5 - 1123389/16384*mu_3*lambda_^2)*z_0 + (((-189/8*mu_3*lambda_^3 - 212320521/131072*mu_3)*eta^2 - 3*mu_3*lambda_^4 - 3370167/16384*mu_3*lambda_)*eta_p + (-1323/32*mu_2*lambda_^3 - 1393119/4096*lambda_^4 - 1486243647/524288*mu_2 - 1565014560291/67108864*lambda_)*eta^4 + (-9/4*lambda_^7 - 10071/512*mu_2*lambda_^3 - 33043383/65536*lambda_^4 - 11313650619/8388608*mu_2 - 12881273688549/536870912*lambda_)*eta + mu_2*lambda_^5 + 1053/128*lambda_^6 + 1123389/16384*mu_2*lambda_^2 + 1182928617/2097152*lambda_^3)*z_1)

Now we have the three common right factors of $L$ with the other generators of Goodearl basis. We have two operators of order 1 and one operator of order 2. How can we compare the solutions?

For the order 1 operators, it is enough to check whether the $2\times 2$ determinant with the coefficients (i.e., whether the operators are proportional or not.

For comparing the order 2 operator with the other two, we need comething else. However, resultants come to rescue: if the resultant is zero, then two operators share a solution. Hence, if we are comparing an order 1 operator with an order 2, then the resultant to be zero is equivalent to the smaller operator divides the bigger or, equivalently, the solution of the order 1 operator is a solution of the order 2 operator.

In [45]:
# Comparing the order 1 operators
det = gcrf_L[0].coefficient_full(z_[0])*gcrf_L[2].coefficient_full(z_[1]) - gcrf_L[2].coefficient_full(z_[0])*gcrf_L[0].coefficient_full(z_[1])
reduce_by_curve(det*z_[0], I, z_)

0

In [46]:
# Comparing the order 2 operator
res = SO_R.sylvester_resultant(gcrf_L[0], gcrf_L[1], z_)
reduce_by_curve(res*z_[0], I, z_)

0

Hence, we conclude there is a common solution to all differential operators in the centralizer of $L$.

### Analyzing the curve in Sage

In [47]:
A.<m_1,m_2,m_3,l> = AffineSpace(QQ,4)
local_dict = {'l':l, 'm_1':m_1, 'm_2':m_2, 'm_3':m_3}
els = [sage_eval(str(el).replace("lambda_", "l").replace("mu_1", "m_1").replace("mu_2", "m_2").replace("mu_3","m_3"), locals=local_dict) for el in I.gens()]
curve = Curve(els, A)
curve.genus()

4